In [2]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GOOGLE_BOOKS_API_KEY")

base_url = 'https://www.googleapis.com/books/v1/'



In [2]:

'''
def getBookData(category, start_Index, index, maxAttempts=3):
    endpoint = f'volumes/?q=subject:{category}&maxResults=40&startIndex={start_Index}&key={api_key}'

    for attempt in range(maxAttempts):
        try:
            res = requests.get(base_url+endpoint)
            response = res.json()

            time.sleep(2)

            item = response.get("items", [])[index] if index < len(response.get("items", [])) else {}

            volume_info = item.get("volumeInfo", {})
            sale_info = item.get("saleInfo", {})
            retail_price_info = sale_info.get("retailPrice", {})

            kind = item.get("kind")
            title = volume_info.get("title")
            date = volume_info.get("publishedDate")
            year = date.split("-")[0] if date else None
            pageCount = volume_info.get("pageCount")
            mature = volume_info.get("maturityRating")
            saleability = sale_info.get("saleability")
            publisher = volume_info.get("publisher")
            authorCount = len(volume_info.get("authors", []))
            retailPrice = retail_price_info.get("amount")
            retailCurrency = retail_price_info.get("currencyCode")
            if res.status_code == 200: 
                return kind, title, date, year, authorCount, pageCount, mature, saleability, publisher,  retailPrice, retailCurrency
            elif res.status_code in [429, 500, 502, 503]:
                time.sleep(2)
            else:
                break
        except:
            requests.exceptions.RequestException
        return None, None, None, None, None, None, None, None,None, None, None



def saveBooksData(categories, book_data_list):
    for category in categories:
        startIndex = 0
        while startIndex < 160:
            for x in range(15):
                kind, title, date, year, authorCount, pageCount, mature, saleability, publisher,  retailPrice, retailCurrency = getBooks(category, startIndex, x)

                row = {
                    "kind": kind,
                    "title": title,
                    "category": category,
                    "publishDate": date,
                    "publishYear": year,
                    "authorCount": authorCount,
                    "pageCount": pageCount,
                    "maturityRating": mature, 
                    "saleability": saleability,
                    "publisher": publisher,
                    "retailPrice": retailPrice,
                    "retailCurrency": retailCurrency
                }

                book_data_list.append(row)
                startIndex = startIndex + 15 if x == 14 else startIndex
    return pd.DataFrame(book_data_list)

'''


'\ndef getBookData(category, start_Index, index, maxAttempts=3):\n    endpoint = f\'volumes/?q=subject:{category}&maxResults=40&startIndex={start_Index}&key={api_key}\'\n\n    for attempt in range(maxAttempts):\n        try:\n            res = requests.get(base_url+endpoint)\n            response = res.json()\n\n            time.sleep(2)\n\n            item = response.get("items", [])[index] if index < len(response.get("items", [])) else {}\n\n            volume_info = item.get("volumeInfo", {})\n            sale_info = item.get("saleInfo", {})\n            retail_price_info = sale_info.get("retailPrice", {})\n\n            kind = item.get("kind")\n            title = volume_info.get("title")\n            date = volume_info.get("publishedDate")\n            year = date.split("-")[0] if date else None\n            pageCount = volume_info.get("pageCount")\n            mature = volume_info.get("maturityRating")\n            saleability = sale_info.get("saleability")\n            publisher

In [3]:
def getBooksData(category, start_Index, maxAttempts=10):
    endpoint = f'volumes/?q=subject:{category}&maxResults=40&startIndex={start_Index}&key={api_key}'

    for attempt in range(maxAttempts):
        try:
            res = requests.get(base_url+endpoint)
            response = res.json()

            time.sleep(2)
            if res.status_code == 200:
                item = response.get("items", [])
                totalItems = response.get("totalItems", 0)
                return totalItems, item
            elif res.status_code in [429, 500, 502, 503]:
                time.sleep(2)
            else:
                break
        except:
            requests.exceptions.RequestException
    return 0, []

def saveBooksData(categories, book_data_list, maxAttempts=10):
    for category in categories:
        startIndex = 0
        while True:
            totalBooks, payload = getBooksData(category, startIndex, maxAttempts=maxAttempts)

            if not payload:
                break 

            for book in payload:
                volume_info = book.get("volumeInfo", {})
                sale_info = book.get("saleInfo", {})
                retail_price_info = sale_info.get("retailPrice", {})

                row = {
                    "kind": book.get("kind"),
                    "title": volume_info.get("title"),
                    "category": category,
                    "publishDate": volume_info.get("publishedDate"),
                    "publishYear": volume_info.get("publishedDate", "").split("-")[0] if volume_info.get("publishedDate") else None,
                    "authorCount": len(volume_info.get("authors", [])),
                    "pageCount": volume_info.get("pageCount"),
                    "maturityRating": volume_info.get("maturityRating"),
                    "saleability": sale_info.get("saleability"),
                    "publisher": volume_info.get("publisher"),
                    "retailPrice": retail_price_info.get("amount"),
                    "retailCurrency": retail_price_info.get("currencyCode")
                }

                book_data_list.append(row)
            startIndex = startIndex + len(payload)

            limit = min(0.5*totalBooks, 175)
            print(f"Fetching {category} | Offset: {startIndex} | Limit: {limit}")
            if startIndex > limit:
                break
    return pd.DataFrame(book_data_list)


In [ ]:
categories = [
    # General & Fiction
    "Fiction",
    "Juvenile+fiction",
    "Poetry",
    "Drama",
    "Comics",
    "Graphic+novels"
    
    # STEM
    "Computers",
    "Science",
    "Mathematics",
    "Technology",
    "Engineering",
    "Medical",
    
    # Humanities & Social Sciences
    "History",
    "Philosophy",
    "Psychology",
    "Law",
    "Education",
    "Language Arts & Disciplines",
    
    # Business & Lifestyle
    "Business",
    "Economics",
    "Biography",
    "Autobiography",
    "Art",
    "Music",
    "Cooking",
    "Health",
    "Fitness",
    "Self-Help",
    "Religion",
    "Travel",
    "Crime",
    "Sports",
]

for category in categories:
    total, payload = getBooksData(category, 0, maxAttempts=20)
    if payload:
        print(f"The {category} category yields a total of {total} book items.")
    


The Fiction category yields a total of 300 book items.
The Juvenile+fiction category yields a total of 300 book items.
The Poetry category yields a total of 300 book items.
The Drama category yields a total of 300 book items.
The Comics category yields a total of 300 book items.
The Science category yields a total of 300 book items.
The Mathematics category yields a total of 300 book items.
The Technology category yields a total of 300 book items.
The Engineering category yields a total of 300 book items.
The Medical category yields a total of 300 book items.
The History category yields a total of 300 book items.
The Philosophy category yields a total of 300 book items.
The Psychology category yields a total of 300 book items.
The Political Science category yields a total of 9 book items.
The Social Science category yields a total of 11 book items.
The Law category yields a total of 300 book items.
The Education category yields a total of 300 book items.
The Language Arts & Disciplines

In [4]:
book_list_1 = []
df = saveBooksData(categories, book_list_1, maxAttempts=20)
df.head()

Fetching Fiction | Offset: 20 | Limit: 150.0
Fetching Fiction | Offset: 40 | Limit: 150.0
Fetching Fiction | Offset: 60 | Limit: 150.0
Fetching Fiction | Offset: 80 | Limit: 150.0
Fetching Fiction | Offset: 100 | Limit: 150.0
Fetching Fiction | Offset: 120 | Limit: 150.0
Fetching Fiction | Offset: 140 | Limit: 150.0
Fetching Fiction | Offset: 160 | Limit: 150.0
Fetching Juvenile+fiction | Offset: 20 | Limit: 150.0
Fetching Poetry | Offset: 20 | Limit: 150.0
Fetching Poetry | Offset: 40 | Limit: 150.0
Fetching Poetry | Offset: 60 | Limit: 150.0
Fetching Poetry | Offset: 80 | Limit: 150.0
Fetching Poetry | Offset: 100 | Limit: 150.0
Fetching Poetry | Offset: 120 | Limit: 150.0
Fetching Poetry | Offset: 140 | Limit: 150.0
Fetching Poetry | Offset: 160 | Limit: 150.0
Fetching Drama | Offset: 20 | Limit: 150.0
Fetching Drama | Offset: 40 | Limit: 150.0
Fetching Drama | Offset: 60 | Limit: 150.0
Fetching Drama | Offset: 80 | Limit: 150.0
Fetching Drama | Offset: 100 | Limit: 150.0
Fetching D

,kind,title,category,publishDate,publishYear,authorCount,pageCount,maturityRating,saleability,publisher,retailPrice,retailCurrency
0,books#volume,Persuasion,Fiction,2023-03-02,2023,1,276.0,NOT_MATURE,FOR_SALE,neobooks,1.49,GBP
1,books#volume,The Face of Another,Fiction,1992,1992,1,260.0,NOT_MATURE,NOT_FOR_SALE,Kodansha,NaN,NaN
2,books#volume,The Street of Crocodiles,Fiction,2025-03-19T00:00:00Z,2025,1,136.0,NOT_MATURE,NOT_FOR_SALE,Rare Treasure Editions,NaN,NaN
3,books#volume,The Sign of the Four Illustrated,Fiction,2021-05-02,2021,1,156.0,NOT_MATURE,NOT_FOR_SALE,NaN,NaN,NaN
4,books#volume,Generations of Winter,Fiction,1994,1994,1,616.0,NOT_MATURE,NOT_FOR_SALE,Random House (NY),NaN,NaN


In [5]:
df.to_csv('../data/books_data_1.csv', index=False)

In [4]:
df = pd.read_csv('../data/books_data_1.csv')

additional_categories = [
    "humor",
    "gardening",
    "music",
    "nature",
    "pets",
    "photography",
    "architecture",
    "politics",
    "spirituality"
]

add_book_list= []
add_df = saveBooksData(additional_categories, add_book_list, maxAttempts= 20)

df_2 = pd.concat([df, add_df], ignore_index=True)
df_2.to_csv('../data/books_data_2.csv', index=False)


Fetching humor | Offset: 20 | Limit: 150.0
Fetching humor | Offset: 40 | Limit: 150.0
Fetching humor | Offset: 60 | Limit: 150.0
Fetching humor | Offset: 80 | Limit: 150.0
Fetching humor | Offset: 100 | Limit: 150.0
Fetching humor | Offset: 120 | Limit: 150.0
Fetching humor | Offset: 140 | Limit: 150.0
Fetching humor | Offset: 160 | Limit: 150.0
Fetching gardening | Offset: 20 | Limit: 150.0
Fetching gardening | Offset: 40 | Limit: 150.0
Fetching gardening | Offset: 60 | Limit: 150.0
Fetching gardening | Offset: 80 | Limit: 150.0
Fetching gardening | Offset: 100 | Limit: 150.0
Fetching gardening | Offset: 120 | Limit: 150.0
Fetching gardening | Offset: 140 | Limit: 150.0
Fetching gardening | Offset: 160 | Limit: 150.0
Fetching music | Offset: 20 | Limit: 150.0
Fetching music | Offset: 40 | Limit: 150.0
Fetching music | Offset: 60 | Limit: 150.0
Fetching music | Offset: 80 | Limit: 150.0
Fetching music | Offset: 100 | Limit: 150.0
Fetching music | Offset: 120 | Limit: 150.0
Fetching mus